# 🚀 LifeMate 2026 (Task 2): Huấn Luyện LoRA DPO Hai Giai Đoạn cho `empero-ai/Qwen3.8-4B-Distill`

Notebook tự động hóa hoàn chỉnh quy trình DPO **không làm phình prompt/schema** (giữ nguyên Champion V15 prompt):
1. **Baseline đã chạy sẵn (`optimize/dpo/baseline_runs/`):** `dev_qwen38_base.jsonl` (**N=300, Mean=0.3940**) & `test_qwen38_base.jsonl` (**N=100, Mean=0.4394**) — Cell 6A tự khôi phục & chấm điểm ngay (~1 giây), **bỏ qua GPU inference** để dành 100% thời gian GPU cho Train DPO.
2. **Giai đoạn 1 (Programmatic DPO — Chạy chính):** Huấn luyện trên tập cân bằng `--max-pairs-per-sample 2 --epochs 1.0` (~2,027 cặp phủ đủ 1,087 mẫu train & 5 nhóm lỗi, 0% rò rỉ Val/Test), tắt `--precompute-ref-log-probs` (PEFT LoRA tự bật/tắt adapter tức thì), stream log từng bước trực tiếp và tự sao lưu/resume checkpoint qua Google Drive.
3. **Đánh giá Trước vs Sau theo App:** Đo trên `dev_ids.txt` (**300 mẫu / 9 app lạ**) và `test_ids.txt` (**100 mẫu / 3 app lạ**).
4. **Giai đoạn 2 (On-Policy Fallback — Nếu Giai đoạn 1 chưa đạt kỳ vọng):** Lấy mẫu rollout trực tiếp từ `empero-ai/Qwen3.8-4B-Distill` qua vLLM trên Colab (`scripts/sample_onpolicy_dpo.py`), bắt lỗi thật của model và train tiếp DPO.

### ⚡ Hướng dẫn chọn Runtime trên Google Colab (`Runtime -> Change runtime type`)
| Runtime Colab | VRAM / Bộ nhớ | `bf16` & CUDA/vLLM | Cấu hình tối ưu mới (`precompute_ref=False`, `max_pairs=2`, `epochs=1.0`) | Thời gian ước tính | Đánh giá |
|---|---:|:---:|---|---|---|
| **G4 (`RTX PRO 6000`)** | **96 GB GDDR7** | ✅ Có (Blackwell) | `bf16` LoRA thuần (tắt 4-bit), `bs=2, grad_accum=8`, TF32+SDPA | **~15–20 phút** (Train) | 🏆 **KHUYÊN DÙNG #1** |
| **NVIDIA L4** | **22.5 GB** | ✅ Có (Ada SM89) | QLoRA 4-bit (`--load-in-4bit`), `bs=1, grad_accum=16`, TF32+SDPA, `--auto-resume` | **~45–60 phút** (Train) | 💰 **LỰA CHỌN #2** |
| **NVIDIA A100** | **40 / 80 GB** | ✅ Có (Ampere SM80) | `bf16` LoRA thuần, `bs=1, grad_accum=16`, TF32+SDPA | **~20–30 phút** (Train) | Rất tốt nhưng hay hết suất |
| **NVIDIA T4** | **15 GB** | ❌ Không (`fp16`) | Dễ OOM / rất chậm với prompt 6K–9K token của DPO | > 4 giờ / OOM | ⛔ **KHÔNG dùng T4** |
| **TPU (`v5e-1` / `v6e-1`)** | 16–32 GB HBM | ❌ Không có CUDA | Stack hiện tại (`trl`/`peft`/`vllm`) yêu cầu NVIDIA CUDA GPU | Không chạy được | ⛔ **KHÔNG chọn TPU** |

📖 Kế hoạch chi tiết xem tại `optimize/dpo/PLAN.md`.

In [ ]:
# 1. Kết nối Google Drive (Lưu checkpoint LoRA DPO, dataset on-policy và đọc data.zip)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/ZAIC_2026_runs/checkpoints', exist_ok=True)
print('✅ Đã kết nối Google Drive thành công!')

In [ ]:
# 2. Kiểm tra GPU & Tự động cấu hình siêu tham số tối ưu theo G4 (96GB) / A100 / L4 / T4
!nvidia-smi || true
import json, torch

assert torch.cuda.is_available(), (
    '❌ Không tìm thấy NVIDIA CUDA GPU (nếu bạn đang chọn TPU v5e-1/v6e-1 hoặc CPU, pipeline trl+vLLM sẽ KHÔNG chạy được). '
    'Hãy vào Runtime -> Change runtime type -> chọn G4 GPU (Khuyên dùng #1) hoặc L4 GPU!'
)
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
bf16_ok = torch.cuda.is_bf16_supported()
print(f'GPU: {gpu_name} | VRAM: {vram_gb:.2f} GB | bf16 supported: {bf16_ok}')

upper_name = gpu_name.upper()
# precompute_ref=False bắt buộc để PEFT LoRA toggle adapter on-the-fly, tránh treo hàng giờ tính ref log-probs trước Step 1
if any(k in upper_name for k in ('PRO 6000', 'BLACKWELL', 'B200', 'G4')) or vram_gb >= 75:
    cfg = {'tier': 'G4-RTX-PRO-6000-96GB', 'load_in_4bit': False, 'bs': 2, 'grad_accum': 8, 'precompute_ref': False, 'gpu_mem': 0.90}
    print('🏆 Phát hiện GPU G4 / >=75GB (96GB GDDR7): Chạy bf16 LoRA thuần tốc độ cao (bs=2, grad_accum=8, precompute_ref=False ~15-20p).')
elif 'A100' in upper_name or vram_gb >= 36:
    cfg = {'tier': 'A100-40GB', 'load_in_4bit': False, 'bs': 1, 'grad_accum': 16, 'precompute_ref': False, 'gpu_mem': 0.90}
    print('✅ Phát hiện GPU A100: Chạy bf16 LoRA thuần (bs=1, grad_accum=16, precompute_ref=False ~20-30p).')
elif 'L4' in upper_name or vram_gb >= 20:
    cfg = {'tier': 'L4-22.5GB', 'load_in_4bit': True, 'bs': 1, 'grad_accum': 16, 'precompute_ref': False, 'gpu_mem': 0.88}
    print('💰 Phát hiện GPU L4 (22.5GB): Bật QLoRA 4-bit + TF32/SDPA + PEFT toggle ref (precompute_ref=False ~45-60p).')
else:
    cfg = {'tier': 'T4-LOW-VRAM', 'load_in_4bit': True, 'bs': 1, 'grad_accum': 16, 'precompute_ref': False, 'gpu_mem': 0.88}
    print('⚠️ CẢNH BÁO: T4 (15GB, không bf16) KHÔNG phù hợp train DPO trên context 9K token (rất dễ OOM / cực chậm).')
    print('👉 Khuyến nghị chuyển Runtime sang G4 (hoặc L4) trước khi chạy Cell 6B!')

with open('/content/dpo_gpu_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)
print('Saved GPU preset -> /content/dpo_gpu_config.json:', cfg)

In [ ]:
# 3. Cài đặt thư viện DPO, Inference & kernel tăng tốc Qwen3.5/Qwen3.8 (flash-linear-attention, causal-conv1d)
import importlib.metadata, importlib.util, subprocess, sys
from packaging.version import Version

# Colab cài sẵn torchao==0.10.0 khiến peft báo ImportError (<0.16.0); gỡ bỏ nếu phiên bản quá cũ
try:
    torchao_ver = importlib.metadata.version('torchao')
    if Version(torchao_ver.split('+')[0]) < Version('0.16.0'):
        print(f'🧹 Gỡ torchao=={torchao_ver} (<0.16.0) do xung đột với peft ...')
        subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
except importlib.metadata.PackageNotFoundError:
    pass

needed = [pkg for pkg, mod in [('trl', 'trl'), ('peft', 'peft'), ('accelerate', 'accelerate'), ('bitsandbytes', 'bitsandbytes'), ('datasets', 'datasets'), ('vllm', 'vllm')] if importlib.util.find_spec(mod) is None]
if needed:
    print(f'⏳ Đang cài đặt: {needed} ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *needed])
else:
    print('✅ Tất cả thư viện lõi (trl, peft, accelerate, bitsandbytes, datasets, vllm) đã có sẵn!')

# Tuỳ chọn: kernel tăng tốc cho kiến trúc lai Gated DeltaNet (qwen3_5) của empero-ai/Qwen3.8-4B-Distill
for opt_pkg, opt_mod in [('flash-linear-attention', 'fla'), ('causal-conv1d', 'causal_conv1d')]:
    if importlib.util.find_spec(opt_mod) is None:
        print(f'⚡ Đang thử cài đặt kernel tuỳ chọn {opt_pkg} ...')
        rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', opt_pkg])
        if rc != 0:
            print(f'ℹ️ Bỏ qua {opt_pkg} (exit={rc}); transformers sẽ tự fallback sang PyTorch ops chuẩn.')

import trl, peft, accelerate, bitsandbytes, transformers, vllm
has_fla = importlib.util.find_spec('fla') is not None
has_conv1d = importlib.util.find_spec('causal_conv1d') is not None
print(f'Versions -> transformers={transformers.__version__}, trl={trl.__version__}, peft={peft.__version__}, accelerate={accelerate.__version__}, bnb={bitsandbytes.__version__}, vllm={vllm.__version__} | fla={has_fla}, causal_conv1d={has_conv1d}')

In [ ]:
# 4. Clone / Pull nhánh planner/vuong của ZAIC_2026 và nạp data/train từ Google Drive
import os, sys
%cd /content
BRANCH = 'planner/vuong'
if not os.path.exists('/content/ZAIC_2026/.git'):
    !git clone https://github.com/Bamboohoccode/ZAIC_2026.git /content/ZAIC_2026

%cd /content/ZAIC_2026
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}
!git log -n 1 --oneline

# Giải nén data.zip vào /content/data/train (giống colab_run_planner.ipynb)
has_local_data = os.path.exists('/content/data/train/train.jsonl') and os.path.exists('/content/data/train/zaic_tools/apps/banking.py')
if not has_local_data:
    zip_candidates = [
        '/content/drive/MyDrive/data.zip',
        '/content/drive/MyDrive/ZAIC_2026/data.zip',
        '/content/drive/MyDrive/ZAIC_2026_runs/data.zip',
        '/content/drive/MyDrive/ZAIC/data.zip',
        '/content/data.zip',
    ]
    found_zip = next((z for z in zip_candidates if os.path.exists(z)), None)
    if found_zip:
        print(f'📦 Giải nén {found_zip} -> /content/ ...')
        !unzip -q -o "{found_zip}" -d /content/
    elif os.path.exists('/content/drive/MyDrive/data/train'):
        os.makedirs('/content/data', exist_ok=True)
        !cp -r /content/drive/MyDrive/data/train /content/data/train
    else:
        raise FileNotFoundError('❌ Không tìm thấy data.zip hoặc data/train trên Google Drive!')

if not os.path.exists('/content/data/train') and os.path.exists('/content/train'):
    os.makedirs('/content/data', exist_ok=True)
    !mv /content/train /content/data/train

os.makedirs('data', exist_ok=True)
!rm -rf data/train
!ln -s /content/data/train data/train
if '/content/data/train' not in sys.path:
    sys.path.insert(0, '/content/data/train')
from zaic_tools import ToolEnv
print('✅ Đã kết nối data/train & zaic_tools.ToolEnv thành công!')

In [ ]:
# 5. Tải model empero-ai/Qwen3.8-4B-Distill & Chạy --dry-run kiểm tra dữ liệu DPO
%cd /content/ZAIC_2026
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
from huggingface_hub import snapshot_download

MODEL_ID = 'empero-ai/Qwen3.8-4B-Distill'
# Nếu repo yêu cầu token cá nhân, điền từ Colab Secrets hoặc biến môi trường HF_TOKEN
hf_token = os.environ.get('HF_TOKEN') or None
try:
    from google.colab import userdata
    hf_token = hf_token or userdata.get('HF_TOKEN')
except Exception:
    pass

local_model_dir = snapshot_download(repo_id=MODEL_ID, token=hf_token)
print(f'✅ Đã tải xong {MODEL_ID} -> {local_model_dir}')
!python -X utf8 -m lifemate.planner.train_dpo --model {MODEL_ID} --wrap-think --max-pairs-per-sample 2 --dry-run

In [ ]:
# 6A. [GIỮ NGUYÊN BASELINE ĐÃ CHẠY] Khôi phục & Kiểm tra điểm Baseline Val (300) & Test (100)
# Nếu đã có đủ 300 mẫu Val & 100 mẫu Test (từ repo optimize/dpo/baseline_runs/ hoặc Google Drive), bỏ qua bước chạy vLLM Baseline để dành GPU train DPO!
%cd /content/ZAIC_2026
import json, os, shutil, sys

cfg = json.load(open('/content/dpo_gpu_config.json'))
MODEL_ID = 'empero-ai/Qwen3.8-4B-Distill'
DRIVE_RUNS = '/content/drive/MyDrive/ZAIC_2026_runs'
REPO_BASE = 'optimize/dpo/baseline_runs'
FORCE_RERUN_BASELINE = False  # Chỉ đặt True nếu muốn tự chạy lại Baseline bằng GPU

os.makedirs('runs', exist_ok=True)
os.makedirs(DRIVE_RUNS, exist_ok=True)

def run_live_stream(cmd_args, log_path=None):
    import os, subprocess, sys
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['HF_HUB_DISABLE_XET'] = '1'
    env['TQDM_MININTERVAL'] = '5'
    print('▶ Command:', ' '.join(cmd_args), flush=True)
    log_f = None
    if log_path:
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        log_f = open(log_path, 'a', encoding='utf-8')
        log_f.write('\n=== RUN: ' + ' '.join(cmd_args) + ' ===\n')
        log_f.flush()
    try:
        proc = subprocess.Popen(
            cmd_args,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            if log_f is not None:
                log_f.write(line)
                log_f.flush()
        rc = proc.wait()
    finally:
        if log_f is not None:
            log_f.close()
    assert rc == 0, f'❌ Lệnh thất bại với exit={rc}'

def count_lines(path):
    if not os.path.exists(path):
        return 0
    with open(path, encoding='utf-8') as f:
        return sum(1 for ln in f if ln.strip())

TARGETS = [('dev_qwen38_base.jsonl', 300, 'eval/splits/dev_ids.txt'), ('test_qwen38_base.jsonl', 100, 'eval/splits/test_ids.txt')]

for fname, expected_n, ids_path in TARGETS:
    dst = f'runs/{fname}'
    if not FORCE_RERUN_BASELINE:
        for cand in [dst, os.path.join(DRIVE_RUNS, fname), os.path.join(REPO_BASE, fname)]:
            if count_lines(cand) >= expected_n:
                if os.path.abspath(cand) != os.path.abspath(dst):
                    shutil.copy2(cand, dst)
                print(f'✅ Đã có sẵn {fname} ({count_lines(dst)}/{expected_n} mẫu) từ {cand} -> Bỏ qua inference!')
                break
    if FORCE_RERUN_BASELINE or count_lines(dst) < expected_n:
        for cand in [os.path.join(DRIVE_RUNS, fname), os.path.join(REPO_BASE, fname)]:
            if os.path.exists(cand) and not os.path.exists(dst):
                shutil.copy2(cand, dst)
        print(f'⏳ Đang chạy Baseline cho {fname} (hiện có {count_lines(dst)}/{expected_n} mẫu)...')
        run_live_stream([
            sys.executable, '-u', '-X', 'utf8', '-m', 'lifemate.planner.runner',
            '--root', 'data', '--split', 'train', '--text',
            '--llm', MODEL_ID, '--mapper', 'none',
            '--gpu-mem', str(cfg['gpu_mem']), '--max-ctx', '12288', '--verify-rounds', '0',
            '--prompt', 'prompts/gepa/vuong/planner_v15.txt',
            '--ids', ids_path, '--resume', '--out', dst,
        ], log_path=f'{DRIVE_RUNS}/runner_baseline.log')

    # Đảm bảo bản hoàn chỉnh cũng nằm trên Drive
    drive_dst = os.path.join(DRIVE_RUNS, fname)
    if count_lines(dst) >= expected_n and count_lines(drive_dst) < expected_n:
        shutil.copy2(dst, drive_dst)

print('\n=== BASELINE VAL (300 mẫu - chia theo app) ===')
!python eval/scorer.py --sub runs/dev_qwen38_base.jsonl --dev-ids eval/splits/dev_ids.txt --by-app
print('\n=== BASELINE TEST (100 mẫu - chia theo app) ===')
!python eval/scorer.py --sub runs/test_qwen38_base.jsonl --dev-ids eval/splits/test_ids.txt --by-app

In [ ]:
# 6B. [GIAI ĐOẠN 1 - TRAIN DPO LIVE STREAM] Huấn luyện LoRA DPO (Tự lưu & Auto-Resume Google Drive)
%cd /content/ZAIC_2026
!git pull origin planner/vuong
import json, os, sys

cfg = json.load(open('/content/dpo_gpu_config.json'))
MODEL_ID = 'empero-ai/Qwen3.8-4B-Distill'
OUT_DIR_S1 = 'checkpoints/planner_dpo_qwen38_lora'
DRIVE_RUNS = '/content/drive/MyDrive/ZAIC_2026_runs'
DRIVE_CKPT_S1 = f'{DRIVE_RUNS}/checkpoints/planner_dpo_qwen38_lora'
LOG_S1 = f'{DRIVE_RUNS}/train_dpo_s1.log'
# Prompt của empero-ai/Qwen3.8-4B-Distill mở sẵn <think>\n -> bật --wrap-think để đóng </think>\n\n sau BƯỚC 1/2/3 đồng bộ ở cả chosen & rejected
WRAP_THINK = True

def run_live_stream(cmd_args, log_path=None):
    import os, subprocess, sys
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['HF_HUB_DISABLE_XET'] = '1'
    env['TQDM_MININTERVAL'] = '5'
    print('▶ Command:', ' '.join(cmd_args), flush=True)
    log_f = None
    if log_path:
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        log_f = open(log_path, 'a', encoding='utf-8')
        log_f.write('\n=== RUN: ' + ' '.join(cmd_args) + ' ===\n')
        log_f.flush()
    try:
        proc = subprocess.Popen(
            cmd_args,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            if log_f is not None:
                log_f.write(line)
                log_f.flush()
        rc = proc.wait()
    finally:
        if log_f is not None:
            log_f.close()
    assert rc == 0, f'❌ Lệnh thất bại với exit={rc}'

cmd = [
    sys.executable, '-u', '-X', 'utf8', '-m', 'lifemate.planner.train_dpo',
    '--model', MODEL_ID,
    '--train-data', 'optimize/dpo/dataset/dpo_train.jsonl',
    '--val-data', 'optimize/dpo/dataset/dpo_val.jsonl',
    '--out-dir', OUT_DIR_S1,
    '--epochs', '1.0', '--lr', '5e-6', '--beta', '0.1', '--rank', '16',
    '--max-pairs-per-sample', '2',
    '--max-prompt-length', '9216', '--max-length', '10240',
    '--bs', str(cfg['bs']), '--grad-accum', str(cfg['grad_accum']),
    '--save-steps', '50', '--eval-steps', '50', '--logging-steps', '2',
    '--auto-resume',
    '--drive-backup-dir', DRIVE_CKPT_S1,
]
if cfg.get('precompute_ref'):
    cmd.append('--precompute-ref-log-probs')
if cfg.get('load_in_4bit'):
    cmd.append('--load-in-4bit')
if WRAP_THINK:
    cmd.append('--wrap-think')

print(f'📝 Log training đang ghi trực tiếp ra Colab + Drive: {LOG_S1}', flush=True)
run_live_stream(cmd, log_path=LOG_S1)
print(f'✅ Hoàn tất Stage 1 LoRA -> {OUT_DIR_S1} (đã đồng bộ lên {DRIVE_CKPT_S1})')

In [ ]:
# 7. [SAU DPO GIAI ĐOẠN 1] Đánh giá Trước vs Sau trên Val (300) & Test (100) chia theo App
%cd /content/ZAIC_2026
import json, os, shutil, sys

cfg = json.load(open('/content/dpo_gpu_config.json'))
MODEL_ID = 'empero-ai/Qwen3.8-4B-Distill'
OUT_DIR_S1 = 'checkpoints/planner_dpo_qwen38_lora'
DRIVE_RUNS = '/content/drive/MyDrive/ZAIC_2026_runs'
DRIVE_CKPT_S1 = f'{DRIVE_RUNS}/checkpoints/planner_dpo_qwen38_lora'

# Tự khôi phục adapter LoRA & baseline nếu Colab vừa mở phiên mới
if not os.path.exists(f'{OUT_DIR_S1}/adapter_config.json') and os.path.exists(f'{DRIVE_CKPT_S1}/adapter_config.json'):
    shutil.copytree(DRIVE_CKPT_S1, OUT_DIR_S1, dirs_exist_ok=True)
os.makedirs('runs', exist_ok=True)
for base_f in ['dev_qwen38_base.jsonl', 'test_qwen38_base.jsonl']:
    if not os.path.exists(f'runs/{base_f}') and os.path.exists(f'optimize/dpo/baseline_runs/{base_f}'):
        shutil.copy2(f'optimize/dpo/baseline_runs/{base_f}', f'runs/{base_f}')

def run_live_stream(cmd_args, log_path=None):
    import os, subprocess, sys
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['HF_HUB_DISABLE_XET'] = '1'
    env['TQDM_MININTERVAL'] = '5'
    print('▶ Command:', ' '.join(cmd_args), flush=True)
    log_f = None
    if log_path:
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        log_f = open(log_path, 'a', encoding='utf-8')
        log_f.write('\n=== RUN: ' + ' '.join(cmd_args) + ' ===\n')
        log_f.flush()
    try:
        proc = subprocess.Popen(
            cmd_args,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            if log_f is not None:
                log_f.write(line)
                log_f.flush()
        rc = proc.wait()
    finally:
        if log_f is not None:
            log_f.close()
    assert rc == 0, f'❌ Lệnh thất bại với exit={rc}'

run_live_stream([
    sys.executable, '-u', '-X', 'utf8', '-m', 'lifemate.planner.runner',
    '--root', 'data', '--split', 'train', '--text',
    '--llm', MODEL_ID, '--llm-lora', OUT_DIR_S1, '--mapper', 'none',
    '--gpu-mem', str(cfg['gpu_mem']), '--max-ctx', '12288', '--verify-rounds', '0',
    '--prompt', 'prompts/gepa/vuong/planner_v15.txt',
    '--ids', 'eval/splits/dev_ids.txt', '--resume', '--out', 'runs/dev_qwen38_dpo_s1.jsonl',
], log_path=f'{DRIVE_RUNS}/eval_dpo_s1_dev.log')

run_live_stream([
    sys.executable, '-u', '-X', 'utf8', '-m', 'lifemate.planner.runner',
    '--root', 'data', '--split', 'train', '--text',
    '--llm', MODEL_ID, '--llm-lora', OUT_DIR_S1, '--mapper', 'none',
    '--gpu-mem', str(cfg['gpu_mem']), '--max-ctx', '12288', '--verify-rounds', '0',
    '--prompt', 'prompts/gepa/vuong/planner_v15.txt',
    '--ids', 'eval/splits/test_ids.txt', '--resume', '--out', 'runs/test_qwen38_dpo_s1.jsonl',
], log_path=f'{DRIVE_RUNS}/eval_dpo_s1_test.log')

print('\n📊 === SO SÁNH VAL (300 MẪU - 9 APP LẠ) ===')
!python eval/scorer.py --sub runs/dev_qwen38_base.jsonl --compare runs/dev_qwen38_dpo_s1.jsonl --dev-ids eval/splits/dev_ids.txt --by-app
print('\n📊 === SO SÁNH TEST (100 MẪU - 3 APP LẠ) ===')
!python eval/scorer.py --sub runs/test_qwen38_base.jsonl --compare runs/test_qwen38_dpo_s1.jsonl --dev-ids eval/splits/test_ids.txt --by-app

# Đồng bộ kết quả sang Google Drive
os.makedirs(DRIVE_RUNS, exist_ok=True)
for fname in os.listdir('runs'):
    if fname.endswith('.jsonl') or fname.endswith('.log'):
        shutil.copy2(f'runs/{fname}', f'{DRIVE_RUNS}/{fname}')
print(f'✅ Đã sao lưu toàn bộ runs/ sang {DRIVE_RUNS}')

---
## 🔄 GIAI ĐOẠN 2 (FALLBACK): Lấy mẫu On-Policy trực tiếp từ empero-ai/Qwen3.8-4B-Distill

**Chỉ chạy 2 cell dưới đây nếu kết quả Giai đoạn 1 chưa đạt kỳ vọng** (ví dụ: Mean Score trên dev_ids.txt tăng < +0.0300 so với baseline, hoặc model vẫn lặp lại các lỗi đặc thù chưa xuất hiện trong bộ programmatic).
- scripts/sample_onpolicy_dpo.py chạy trực tiếp empero-ai/Qwen3.8-4B-Distill trên eval/splits/dpo_train_ids.txt (1,145 mẫu — tự động loại 57 mẫu thuộc dpo_val.jsonl để đảm bảo 0% rò rỉ), kiểm chứng qua ToolEnv và ghép cặp (chosen, rejected) từ **chính câu trả lời sai thực tế của Qwen 3.8 4B**.


In [ ]:
# 8A. [GIAI ĐOẠN 2 - BƯỚC 1] Lấy mẫu On-Policy Rollouts trực tiếp từ empero-ai/Qwen3.8-4B-Distill
%cd /content/ZAIC_2026
import json, os, shutil, sys

cfg = json.load(open('/content/dpo_gpu_config.json'))
MODEL_ID = 'empero-ai/Qwen3.8-4B-Distill'
DRIVE_RUNS = '/content/drive/MyDrive/ZAIC_2026_runs'

# Đặt SAMPLE_WITH_S1_LORA = True nếu muốn mine lỗi còn sót SAU Giai đoạn 1 (iterative DPO); False để mine lỗi từ Base Qwen 3.8 4B
SAMPLE_WITH_S1_LORA = False
ONPOLICY_PATH = 'optimize/dpo/dataset/dpo_onpolicy_train.jsonl'

def run_live_stream(cmd_args, log_path=None):
    import os, subprocess, sys
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['HF_HUB_DISABLE_XET'] = '1'
    env['TQDM_MININTERVAL'] = '5'
    print('▶ Command:', ' '.join(cmd_args), flush=True)
    log_f = None
    if log_path:
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        log_f = open(log_path, 'a', encoding='utf-8')
        log_f.write('\n=== RUN: ' + ' '.join(cmd_args) + ' ===\n')
        log_f.flush()
    try:
        proc = subprocess.Popen(
            cmd_args,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            if log_f is not None:
                log_f.write(line)
                log_f.flush()
        rc = proc.wait()
    finally:
        if log_f is not None:
            log_f.close()
    assert rc == 0, f'❌ Lệnh thất bại với exit={rc}'

cmd = [
    sys.executable, '-u', '-X', 'utf8', 'scripts/sample_onpolicy_dpo.py',
    '--root', 'data', '--split', 'train',
    '--ids', 'eval/splits/dpo_train_ids.txt',
    '--prompt', 'prompts/gepa/vuong/planner_v15.txt',
    '--model', MODEL_ID,
    '--num-samples', '3', '--batch-size', '16',
    '--gpu-mem', str(cfg['gpu_mem']), '--max-ctx', '12288',
    '--think-mode', 'prefix', '--include-programmatic', '--resume',
    '--out', ONPOLICY_PATH,
]
if SAMPLE_WITH_S1_LORA:
    cmd.extend(['--lora', 'checkpoints/planner_dpo_qwen38_lora'])

run_live_stream(cmd, log_path=f'{DRIVE_RUNS}/sample_onpolicy.log')

os.makedirs(DRIVE_RUNS, exist_ok=True)
drive_ds = f'{DRIVE_RUNS}/dpo_onpolicy_train.jsonl'
shutil.copy2(ONPOLICY_PATH, drive_ds)
print(f'✅ Đã lưu dữ liệu On-Policy -> {ONPOLICY_PATH} và sao lưu sang {drive_ds}')

In [ ]:
# 8B. [GIAI ĐOẠN 2 - BƯỚC 2] Train tiếp LoRA DPO trên On-Policy Rollouts & Chấm điểm so sánh
%cd /content/ZAIC_2026
import json, os, shutil, sys

cfg = json.load(open('/content/dpo_gpu_config.json'))
MODEL_ID = 'empero-ai/Qwen3.8-4B-Distill'
ONPOLICY_PATH = 'optimize/dpo/dataset/dpo_onpolicy_train.jsonl'
OUT_DIR_S2 = 'checkpoints/planner_dpo_qwen38_s2_lora'
DRIVE_RUNS = '/content/drive/MyDrive/ZAIC_2026_runs'
DRIVE_CKPT_S2 = f'{DRIVE_RUNS}/checkpoints/planner_dpo_qwen38_s2_lora'

def run_live_stream(cmd_args, log_path=None):
    import os, subprocess, sys
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['HF_HUB_DISABLE_XET'] = '1'
    env['TQDM_MININTERVAL'] = '5'
    print('▶ Command:', ' '.join(cmd_args), flush=True)
    log_f = None
    if log_path:
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        log_f = open(log_path, 'a', encoding='utf-8')
        log_f.write('\n=== RUN: ' + ' '.join(cmd_args) + ' ===\n')
        log_f.flush()
    try:
        proc = subprocess.Popen(
            cmd_args,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            if log_f is not None:
                log_f.write(line)
                log_f.flush()
        rc = proc.wait()
    finally:
        if log_f is not None:
            log_f.close()
    assert rc == 0, f'❌ Lệnh thất bại với exit={rc}'

cmd = [
    sys.executable, '-u', '-X', 'utf8', '-m', 'lifemate.planner.train_dpo',
    '--model', MODEL_ID,
    '--train-data', ONPOLICY_PATH,
    '--val-data', 'optimize/dpo/dataset/dpo_val.jsonl',
    '--wrap-think',
    '--out-dir', OUT_DIR_S2,
    '--epochs', '1.0', '--lr', '2e-6', '--beta', '0.1', '--rank', '16',
    '--max-prompt-length', '9216', '--max-length', '10240',
    '--bs', str(cfg['bs']), '--grad-accum', str(cfg['grad_accum']),
    '--save-steps', '50', '--eval-steps', '50', '--logging-steps', '2',
    '--auto-resume',
    '--drive-backup-dir', DRIVE_CKPT_S2,
]
if os.path.exists('checkpoints/planner_dpo_qwen38_lora/adapter_config.json'):
    cmd.extend(['--resume-from-lora', 'checkpoints/planner_dpo_qwen38_lora'])
if cfg.get('precompute_ref'):
    cmd.append('--precompute-ref-log-probs')
if cfg.get('load_in_4bit'):
    cmd.append('--load-in-4bit')

run_live_stream(cmd, log_path=f'{DRIVE_RUNS}/train_dpo_s2.log')

run_live_stream([
    sys.executable, '-u', '-X', 'utf8', '-m', 'lifemate.planner.runner',
    '--root', 'data', '--split', 'train', '--text',
    '--llm', MODEL_ID, '--llm-lora', OUT_DIR_S2, '--mapper', 'none',
    '--gpu-mem', str(cfg['gpu_mem']), '--max-ctx', '12288', '--verify-rounds', '0',
    '--prompt', 'prompts/gepa/vuong/planner_v15.txt',
    '--ids', 'eval/splits/dev_ids.txt', '--resume', '--out', 'runs/dev_qwen38_dpo_s2.jsonl',
], log_path=f'{DRIVE_RUNS}/eval_dpo_s2_dev.log')

run_live_stream([
    sys.executable, '-u', '-X', 'utf8', '-m', 'lifemate.planner.runner',
    '--root', 'data', '--split', 'train', '--text',
    '--llm', MODEL_ID, '--llm-lora', OUT_DIR_S2, '--mapper', 'none',
    '--gpu-mem', str(cfg['gpu_mem']), '--max-ctx', '12288', '--verify-rounds', '0',
    '--prompt', 'prompts/gepa/vuong/planner_v15.txt',
    '--ids', 'eval/splits/test_ids.txt', '--resume', '--out', 'runs/test_qwen38_dpo_s2.jsonl',
], log_path=f'{DRIVE_RUNS}/eval_dpo_s2_test.log')

print('\n📊 === SO SÁNH BASELINE vs STAGE 2 ON-POLICY DPO (VAL 300) ===')
!python eval/scorer.py --sub runs/dev_qwen38_base.jsonl --compare runs/dev_qwen38_dpo_s2.jsonl --dev-ids eval/splits/dev_ids.txt --by-app
print('\n📊 === SO SÁNH BASELINE vs STAGE 2 ON-POLICY DPO (TEST 100) ===')
!python eval/scorer.py --sub runs/test_qwen38_base.jsonl --compare runs/test_qwen38_dpo_s2.jsonl --dev-ids eval/splits/test_ids.txt --by-app

---
## 📦 9. Xuất & Tải Trọng Số Đã Fine-Tune (LoRA ~50MB / Google Drive / HuggingFace Hub)

Sau khi train xong, bạn **KHÔNG cần tải lại toàn bộ 8 GB của model gốc** — chỉ cần lấy thư mục **LoRA adapter (~50–65 MB)** là chạy được ngay trên Kaggle / Colab / Docker A30 bằng cờ --llm empero-ai/Qwen3.8-4B-Distill --llm-lora <path>.
Cell này hỗ trợ cả 3 cách lấy weight:
1. **Google Drive (đã tự đồng bộ sau Cell 6B / 8B):** /content/drive/MyDrive/ZAIC_2026_runs/checkpoints/...
2. **Tải trực tiếp file .zip về máy từ trình duyệt** (DOWNLOAD_IN_BROWSER = True)
3. **Tuỳ chọn Merge sang 4B thuần hoặc đẩy lên HuggingFace Hub** (MERGE_FULL_MODEL = True / PUSH_TO_HF = True)

In [ ]:
# 9. Đóng gói .zip, tải LoRA về máy, hoặc tuỳ chọn Merge / Push HuggingFace Hub
%cd /content/ZAIC_2026
import os, shutil
from pathlib import Path

MODEL_ID = 'empero-ai/Qwen3.8-4B-Distill'
DRIVE_CKPTS = Path('/content/drive/MyDrive/ZAIC_2026_runs/checkpoints')
CKPT_S2 = Path('checkpoints/planner_dpo_qwen38_s2_lora')
CKPT_S1 = Path('checkpoints/planner_dpo_qwen38_lora')
for local_p, drive_p in [(CKPT_S2, DRIVE_CKPTS / CKPT_S2.name), (CKPT_S1, DRIVE_CKPTS / CKPT_S1.name)]:
    if not (local_p / 'adapter_config.json').exists() and (drive_p / 'adapter_config.json').exists():
        shutil.copytree(drive_p, local_p, dirs_exist_ok=True)
CKPT_DIR = CKPT_S2 if (CKPT_S2 / 'adapter_config.json').exists() else CKPT_S1
assert (CKPT_DIR / 'adapter_config.json').exists(), '❌ Chưa tìm thấy checkpoint LoRA nào (cả local lẫn Google Drive)!'

# 9.1 Nén adapter LoRA thành .zip & sao lưu Drive
zip_base = Path('/content') / CKPT_DIR.name
zip_file = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=CKPT_DIR.parent, base_dir=CKPT_DIR.name))
size_mb = zip_file.stat().st_size / (1024 ** 2)
print(f'✅ Đã đóng gói LoRA adapter ({size_mb:.1f} MB): {zip_file}')

drive_zip = Path('/content/drive/MyDrive/ZAIC_2026_runs') / zip_file.name
if drive_zip.parent.exists():
    shutil.copy(zip_file, drive_zip)
    print(f'✅ Đã lưu bản sao lưu .zip lên Google Drive: {drive_zip}')

DOWNLOAD_IN_BROWSER = True
if DOWNLOAD_IN_BROWSER:
    try:
        from google.colab import files
        files.download(str(zip_file))
    except Exception as e:
        print('ℹ️ Không mở được tải trình duyệt (có thể lấy file trực tiếp từ Drive):', e)

# 9.2 (Tuỳ chọn) Merge LoRA + Base thành model 4B hợp nhất (~8GB) hoặc Push HuggingFace Hub
MERGE_FULL_MODEL = False
MERGED_DIR = Path('checkpoints/qwen38_4b_dpo_merged')
if MERGE_FULL_MODEL:
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print('⏳ Đang merge LoRA adapter vào empero-ai/Qwen3.8-4B-Distill ...')
    base_m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, trust_remote_code=True)
    merged_m = PeftModel.from_pretrained(base_m, str(CKPT_DIR)).merge_and_unload()
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    merged_m.save_pretrained(str(MERGED_DIR), safe_serialization=True)
    AutoTokenizer.from_pretrained(str(CKPT_DIR), trust_remote_code=True).save_pretrained(str(MERGED_DIR))
    print(f'✅ Đã lưu model hợp nhất -> {MERGED_DIR}')

PUSH_TO_HF = False
HF_TARGET_REPO = ''  # Ví dụ: 'ten-cua-ban/qwen38-4b-lifemate-dpo-lora'
if PUSH_TO_HF:
    assert HF_TARGET_REPO, 'Hãy điền HF_TARGET_REPO trước khi bật PUSH_TO_HF!'
    from huggingface_hub import HfApi
    upload_src = MERGED_DIR if MERGE_FULL_MODEL else CKPT_DIR
    hf_tok = os.environ.get('HF_TOKEN')
    api = HfApi(token=hf_tok)
    api.create_repo(HF_TARGET_REPO, exist_ok=True, private=True)
    api.upload_folder(folder_path=str(upload_src), repo_id=HF_TARGET_REPO)
    print(f'✅ Đã đẩy trọng số lên HuggingFace Hub: https://huggingface.co/{HF_TARGET_REPO}')